### Initial attempt by D. Barco on Thu 25-Sep-2025 @ 23h45
### completed assignment-1 (without complete testing) on Tue 30-Sep-2025 @ 14h20
##### Experiencing various dask import errors & kernel crashes. Troubleshooting with Xindi, Dmytro, Jesús during Friday & Saturday office hours. Dmytro resolved the dask import error by having me redo the 'dsi_participant' environment in conda. Sat 27-Sep-2025 @ 09h00 - 10h20
##### Note: we did not install the last 3 pip packages (l0bnb faraway islp) as Dmytro thinks there is a conflict between conda and pip
### retried assignment-1 with 'dsi_production_only' kernel @16h45
### seems to run ok now! (no crashes)

# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv

In [2]:
import dask.dataframe as dd

In [ ]:
################################################################
### flwg section courtesy of Dmytro via Slack
### -- added Tue 30-Sep-2025 @ 14h00
#import dask
#dask.config.set(scheduler="single-threaded") 

### Limit Dask to just one thread, so it won’t overload your processor
#from dask.distributed import Client
#client = Client(n_workers=1, threads_per_worker=1)
################################################################

2025-09-30 14:59:13,826 - distributed.nanny - WARNING - Restarting worker
2025-09-30 14:59:16,802 - distributed.nanny - WARNING - Restarting worker
2025-09-30 14:59:19,770 - distributed.nanny - WARNING - Restarting worker
2025-09-30 14:59:22,716 - distributed.scheduler - ERROR - Task ('read_parquet-40581a0084f76b4e36040ddce80a17ce', 176) marked as failed because 4 workers died while trying to run it
2025-09-30 14:59:22,716 - distributed.scheduler - ERROR - Task ('read_parquet-40581a0084f76b4e36040ddce80a17ce', 178) marked as failed because 4 workers died while trying to run it
2025-09-30 14:59:22,772 - distributed.nanny - WARNING - Restarting worker
2025-09-30 14:59:58,232 - distributed.nanny - WARNING - Restarting worker
2025-09-30 15:00:01,178 - distributed.nanny - WARNING - Restarting worker
2025-09-30 15:00:04,221 - distributed.nanny - WARNING - Restarting worker
2025-09-30 15:00:07,099 - distributed.scheduler - ERROR - Task ('read_parquet-40581a0084f76b4e36040ddce80a17ce', 176) ma

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
import sys
sys.path.append(os.getenv('PRICE_DATA'))

# parquet_files = glob((os.path.join(os.getenv('PRICE_DATA'), "data/temp/parquet/*.parquet"))) # <-- ignore (explicit path)
parquet_files = glob(os.path.join(os.getenv('PRICE_DATA'), "**/*.parquet"), recursive = True)

dd_px = (dd
         .read_parquet(parquet_files)   # <-- "broken up" for readability
         .set_index("ticker"))

# stopped here Fri 26-Sep-2025 @ 00h56
# continuing Tue 30-Sep-2025 @ 11h45

In [4]:
dd_px.info() # <-- IGNORE (just checking)

<class 'dask_expr.DataFrame'>
Columns: 9 entries, Date to Year
dtypes: datetime64[ns](1), float64(5), int32(1), int64(1), string(1)

### [Aside: https://docs.dask.org/en/stable/dataframe-best-practices.html ... my kernel keeps crashing with Dask, perhaps I should use Pandas DataFrame instead??]

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.

# Add lags for variables 'Close' & 'Adj Close' (grouped by 'ticker')
# The code below is structured for multiple columns i.e. 'Close_lag_1' & 'Adj_Close_lag_1'
# code reference: https://docs.dask.org/en/stable/generated/dask.dataframe.DataFrame.assign.html
# Note: I cannot test it properly because the kernel keeps crashing!! :-(
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),
                       Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

# following code block will append the 'returns' column
dd_rets = dd_shift.assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1 # <-- calculate returns
)

# following code block will append the 'hi_lo_range' column
dd_feat = dd_rets.assign(
    hi_lo_range = lambda x: x['High'] - x['Low'] # <-- calculate high-low spread
)

dd_feat.info() # <-- IGNORE (just checking)

<class 'dask_expr.DataFrame'>
Columns: 13 entries, Date to hi_lo_range
dtypes: datetime64[ns](1), float64(9), int32(1), int64(1), string(1)

C:\Users\D_User\AppData\Local\Temp\ipykernel_20808\1740471529.py:7: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [7]:
# Write your code below.

pd_feat = dd_feat.compute() # <-- need to materialize because of the lazy execution nature
                            # of Dask DataFrames
# From lecture, materializing a Dask DataFrame will convert it to a Pandas DataFrame

# calculate 10-day moving average of 'returns'
pd_feat['moving_avg_returns_10d'] = pd_feat['returns'].rolling(10).mean()

In [ ]:
dd_px # <-- (just checking) (no more errors!)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=35,,,,,,,,,
AAMC,datetime64[ns],float64,float64,float64,float64,float64,int64,string,int32
AIM,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZUMZ,...,...,...,...,...,...,...,...,...
ZUMZ,...,...,...,...,...,...,...,...,...


In [9]:
pd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,moving_avg_returns_10d
ticker,,,,,,,,,,,,,,
AAMC,2012-12-13,15.000000,15.000000,15.00,15.000000,15.000000,100.0,AAMC.csv,2012,NaN,NaN,NaN,0.000000,NaN
AAMC,2012-12-14,19.000000,30.000000,19.00,30.000000,30.000000,144600.0,AAMC.csv,2012,15.000000,15.000000,1.000000,11.000000,NaN
AAMC,2012-12-17,31.500000,65.000000,31.50,65.000000,65.000000,68600.0,AAMC.csv,2012,30.000000,30.000000,1.166667,33.500000,NaN
AAMC,2012-12-18,65.000000,89.000000,65.00,80.000000,80.000000,43600.0,AAMC.csv,2012,65.000000,65.000000,0.230769,24.000000,NaN
AAMC,2012-12-19,80.000000,84.000000,78.00,84.000000,84.000000,24000.0,AAMC.csv,2012,80.000000,80.000000,0.050000,6.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZUMZ,2020-03-26,19.450001,19.450001,18.25,18.639999,18.639999,430500.0,ZUMZ.csv,2020,19.350000,19.350000,-0.036693,1.200001,0.004211
ZUMZ,2020-03-27,17.920000,18.570000,17.23,17.469999,17.469999,369600.0,ZUMZ.csv,2020,18.639999,18.639999,-0.062768,1.340000,-0.006423
ZUMZ,2020-03-30,17.570000,17.670000,16.52,17.430000,17.430000,486700.0,ZUMZ.csv,2020,17.469999,17.469999,-0.002290,1.150000,-0.003729


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

No it is not necessary, however, from my understanding from the course material, materializing the Dask DataFrame automatically converts it into a Pandas DataFrame.

+ Would it have been better to do it in Dask? Why?

For large datasets, it is preferred to do the operation in Dask because we can take advantage of Dask's distributed computing capability. However, this is all speculation for me as my kernel keeps crashing. (or see 'KilledWorker' error above.) P.S. I will attempt to create a different environment setup for Production Assignment-2 if needed. P.P.S. So it looks like the dedicated environment setup 'dsi_production_only (Python 3.9.23)' worked! :-)

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.